# 07 Final Evaluation And Figures

Purpose: execute the 07 final evaluation and figures stage.
Required inputs: DATA_ROOT plus artifacts from prior stages when listed.
Expected outputs: generation_summary.csv, reconstruction/morphology/diversity CSVs, figures.
Fold: FOLD; seed: SEED; all unexecuted results remain `not_run`.


In [ ]:
from pathlib import Path
import os, sys, platform, re, torch

print("=== CHECKING /kaggle/input ===")
input_dir = Path("/kaggle/input")
if input_dir.exists():
    items = list(input_dir.iterdir())
    print(f"Contents of /kaggle/input ({len(items)} items): {[p.name for p in items]}")
    for it in items:
        if it.is_dir():
            print(f"  Folder {it.name} contains {len(list(it.iterdir()))} items")
else:
    print("/kaggle/input does not exist!")

# Robust discovery of DATA_ROOT
DATA_ROOT = None
if os.environ.get('PPGDALIA_ROOT'):
    p = Path(os.environ['PPGDALIA_ROOT'])
    if p.exists():
        DATA_ROOT = p

if DATA_ROOT is None and input_dir.exists():
    for p in input_dir.rglob("*.pkl"):
        if re.search(r'(?<![A-Z0-9])S(1[0-5]|[1-9])(?![A-Z0-9])', p.stem.upper()):
            rel = p.relative_to(input_dir)
            DATA_ROOT = input_dir / rel.parts[0]
            print(f"Auto-discovered DATA_ROOT via subject files: {DATA_ROOT}")
            break

if DATA_ROOT is None:
    _candidates = [
        Path('/kaggle/input/ppg-dalia-dataset'),
        Path('/kaggle/input/ppg-dalia'),
        Path('/kaggle/input/datasets/ameersifat53/ppg-dalia-dataset')
    ]
    DATA_ROOT = next((p for p in _candidates if p.exists() and p.is_dir()), _candidates[0])

# Project-relative output is collected by krun.yaml outputs: artifacts/.
OUTPUT_ROOT = Path(os.environ.get('PPGCVAE_OUTPUT_ROOT', 'artifacts'))
FOLD = 1
SEED = 42
print({'DATA_ROOT': str(DATA_ROOT), 'OUTPUT_ROOT': str(OUTPUT_ROOT), 'device': 'cuda' if torch.cuda.is_available() else 'cpu', 'python': sys.version, 'torch': torch.__version__})
# This stage consumes prepared arrays/checkpoints; DATA_ROOT is not required.


In [ ]:
# Locate this repository without assuming a Kaggle dataset slug.
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'Final_IOT_for_AI']
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository src/ directory; upload/clone the repository first.')
sys.path.insert(0, str(repo_root / 'src'))
from ppg_cvae.data.splits import get_fold
print(get_fold(FOLD))

# Separate Kaggle jobs do not share their writable files automatically.
# If prior artifacts are mounted under /kaggle/input, set this path explicitly.
ARTIFACT_INPUT_ROOT = os.environ.get('PPGCVAE_INPUT_ROOT')
if ARTIFACT_INPUT_ROOT:
    import shutil
    source_root = Path(ARTIFACT_INPUT_ROOT)
    if not source_root.is_dir(): raise FileNotFoundError(source_root)
    if source_root.resolve() != OUTPUT_ROOT.resolve():
        OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
        for folder in ('prepared','proxy','tuning','cvae','gaussian','ablations'):
            source = source_root/folder
            if source.is_dir(): shutil.copytree(source,OUTPUT_ROOT/folder,dirs_exist_ok=True)
    print('Imported previous artifacts from:', source_root)


In [ ]:
from ppg_cvae.workflow import evaluate_run
# Set True only after fixing label alignment, thresholds, detector, proxy policy,
# tuning candidates and selected configs for every fold. Do not tune after test.
EVALUATION_PROTOCOL_LOCKED = False
METHODS = ('cvae','gaussian','no_hr_loss','no_condition')
if not EVALUATION_PROTOCOL_LOCKED:
    raise ValueError('Finalize the evaluation protocol for all folds before opening test data, then set EVALUATION_PROTOCOL_LOCKED=True.')
for fold in range(1,6):
    assert (OUTPUT_ROOT/'tuning'/f'fold_{fold:02d}'/'selected_config.json').exists(), f'Missing selected config for fold {fold}'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for method in METHODS:
    if method in ('no_hr_loss','no_condition') and SEED != 42: continue
    print(evaluate_run(OUTPUT_ROOT,FOLD,SEED,method,device=device).to_string(index=False))
